# Heaps in Python: Learn by Running Code
### A practical classroom notebook: `heapq`, minimums, Top-K, and merging sorted lists

**How to use this:** read a short explanation, run the code directly below it, change the input, and run it again. The notebook is designed for teaching through demonstrations rather than long theory.

Why Do We Need Heaps?

Imagine you are building a shopping application. A customer wants to see the three cheapest products. Your program has thousands of product prices and needs to select the cheapest ones. Or imagine a task-processing system where the next task should be the one with the highest priority. Or a system that combines several already-sorted streams of data.

These problems have something in common: we repeatedly need to identify the most suitable item without doing unnecessary work.

One straightforward solution is to sort all the data and select the item we need. Sorting works, but it may do more work than necessary when we only need a few items.

A heap is a data structure that helps us manage these situations efficiently. A min-heap keeps the smallest value at its root, while a max-heap conceptually keeps the largest value at its root.

In this notebook, we will start with a simple question: how do we find the smallest number in a list? We will solve it using Python's built-in tools, learn how a min-heap works, and compare the approaches using Big-O notation.

By the end, you should understand not just how to write heap code, but what problem a heap solves, when to use one, and when a simpler solution is better.

## 1. Our starting problem

Find the smallest number in this list. Python's built-in `min()` is the simplest baseline and takes **O(n)** time because it may inspect every value.

In [1]:
numbers = [20, 5, 10, 12, 2, 1, 7, 11, 18, 12]
print("Smallest:", min(numbers))

Smallest: 1


## 2. What is a min-heap?

A **min-heap** keeps each parent less than or equal to its children. That guarantees the smallest value is at the root. A heap is **not fully sorted**; only the parent-child rule is guaranteed. Python's `heapq` stores the heap in an ordinary list.

In [2]:
import heapq

numbers = [20, 5, 10, 12, 2, 1, 7, 11, 18, 12]
heapq.heapify(numbers)  # Rearranges this list in-place              #  O(n)time complexity
print("Heap representation:", numbers)
print("Smallest at root:", numbers[0])                               #  O(1) time complexity

Heap representation: [1, 2, 7, 11, 5, 10, 20, 12, 18, 12]
Smallest at root: 1


**Big-O:** `heapify()` builds a heap in **O(n)** time; reading `heap[0]` is **O(1)**. `heapify()` changes the list in place, so copy it first if you need the original list unchanged.

In [7]:
original = [20, 5, 10, 12, 2, 1, 7, 11, 18, 12, 0]
heap = original.copy()
heapq.heapify(heap)
print("Original:", original)
print("Heap:", heap)
print("Minimum:", heap[0])

Original: [20, 5, 10, 12, 2, 1, 7, 11, 18, 12, 0]
Heap: [0, 2, 1, 11, 5, 10, 7, 12, 18, 12, 20]
Minimum: 0


### Quick prediction

What will print? Predict first, then run the cell.

In [20]:
values = [8, 3, 6, 1, 9, 0]
heapq.heapify(values)                       #O(n) time complexity
print(values[0])                            #O(1) time complexity

0


## 3. Push, peek, and pop

- `heappush(heap, value)` adds an item.
- `heap[0]` reads the minimum without removing it.
- `heappop(heap)` removes and returns the minimum.

Push and pop take **O(log n)** time because the heap may need to adjust along its height.

In [24]:
heap = [1, 2, 1, 7, 1, 1, 3, 2]
# heap = [(1, 2), (1, 7), (1, 1), (3, 2)]

heapq.heapify(heap)                             #O(n) time complexity

heapq.heappush(heap, 0)                         #O(log n) time complexity
# heapq.heappush(heap, (2, 1))                  #O(log n) time complexity

print("After push:", heap)
print("Minimum (peek):", heap[0])               #O(1) time complexity
print("Popped minimum:", heapq.heappop(heap))   #O(log n) time complexity
print("Heap after pop:", heap)

After push: [0, 1, 1, 1, 2, 1, 3, 7, 2]
Minimum (peek): 0
Popped minimum: 0
Heap after pop: [1, 1, 1, 1, 2, 2, 3, 7]


### Get the three smallest values

Repeated `heappop()` calls return values from smallest to largest. This is useful when you only need a few small values.

In [ ]:
# values = [20, 5, 10, 12, 2, 1, 7, 11, 18, 12, 0]
values = [(1, 2), (1, 7), (1, 1), (3, 2)]
heapq.heapify(values)
first_three = [heapq.heappop(values) for _ in range(3)]         #O(nlogn) time complexity
print("Three smallest:", first_three)

Three smallest: [(1, 1), (1, 2), (1, 7)]


## 4. Big-O: when is a heap faster?

| Task | Time | Explanation |
|---|---:|---|
| `min(values)` | O(n) | Inspect values to find the minimum |
| `heapify(values)` | O(n) | Build the heap |
| `heap[0]` after heapify | O(1) | Read the root |
| `heappush` / `heappop` | O(log n) | Restore heap order |
| `sorted(values)` | O(n log n) | Sort all values |

**Honest takeaway:** a heap is not asymptotically faster for one minimum in an unsorted list. `min()` is O(n), and heap construction is also O(n). A heap becomes useful when you need repeated minimums or only a few smallest values. Sorting everything costs O(n log n), while building a heap and popping a fixed number `k` costs O(n + k log n).

Run this small timing demo. Results depend on your machine, so treat them as an illustration, not a proof.

In [ ]:
import random
import timeit

data = [random.randint(0, 1_000_000) for _ in range(30_000)]

def heap_min():
    h = data.copy()
    heapq.heapify(h)
    return h[0]

def heap_three():
    h = data.copy()
    heapq.heapify(h)
    return [heapq.heappop(h) for _ in range(3)]

print("min(data), 10 runs:", round(timeit.timeit(lambda: min(data), number=10), 4))
print("heapify + root, 10 runs:", round(timeit.timeit(heap_min, number=10), 4))
print("sort and take 3, 3 runs:", round(timeit.timeit(lambda: sorted(data)[:3], number=3), 4))
print("heapify and pop 3, 3 runs:", round(timeit.timeit(heap_three, number=3), 4))

**Discuss:** compare the two 10-run measurements with each other, and the two 3-run measurements with each other. Heap construction and copying are real costs and must be included.

## 5. Top-K: find the three largest or smallest

Top-K means the `k` highest or lowest values. `heapq.nlargest()` and `heapq.nsmallest()` are convenient when you only need a few values.

In [ ]:
numbers = [20, 5, 10, 12, 2, 1, 7, 11, 18, 12]
print("Three largest:", heapq.nlargest(3, numbers))
print("Three smallest:", heapq.nsmallest(3, numbers))
print("Sorting baseline:", sorted(numbers, reverse=True)[:3])

### Top-K frequent is a different task

Top-K largest ranks by numeric value. Top-K frequent ranks by how often a value appears. Use `Counter` to count, then `most_common()` for a straightforward solution.

In [ ]:
from collections import Counter

numbers = [4, 4, 4, 2, 2, 7, 7, 7, 7, 9, 9, 1]
counts = Counter(numbers)
print("Counts:", counts)
print("Two most frequent (value, count):", counts.most_common(2))

### A bounded min-heap for Top-K frequent

If there are `m` distinct values and we only want `k`, a bounded heap can keep only `k` candidates. Counting takes O(n) average time; heap work is O(m log k), plus O(k log k) to order the final candidates. This can be better than sorting all `m` distinct values when `k` is small.

This example breaks frequency ties by numeric value for predictable output.

In [ ]:
from collections import Counter
import heapq

def top_k_frequent(numbers, k):
    if k <= 0:
        return []
    counts = Counter(numbers)
    heap = []
    for value, frequency in counts.items():
        heapq.heappush(heap, (frequency, value))
        if len(heap) > k:
            heapq.heappop(heap)
    return sorted(heap, key=lambda pair: (-pair[0], pair[1]))

numbers = [4, 4, 4, 2, 2, 7, 7, 7, 7, 9, 9, 1]
print(top_k_frequent(numbers, 2))  # (frequency, value)

**Try it:** make `5` appear five times, `3` appear three times, and `8` appear twice. Ask for the top two frequent values.

## 6. Merge k sorted lists

Suppose each input list is already sorted. A baseline is to combine everything and sort again. A heap-based merge instead compares the smallest currently available value from each list.

In [ ]:
lists = [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
print("Combine and sort:", sorted(value for row in lists for value in row))

### Heap merge, step by step

Put the first item of each non-empty list in a heap. Pop the smallest, append it to the result, and push the next item from that same list. Store `(value, list_index, item_index)` so equal values are safely comparable.

In [ ]:
import heapq

def merge_k_sorted_lists(sorted_lists):
    heap = []
    for list_index, values in enumerate(sorted_lists):
        if values:
            heapq.heappush(heap, (values[0], list_index, 0))

    result = []
    while heap:
        value, list_index, item_index = heapq.heappop(heap)
        result.append(value)
        next_index = item_index + 1
        if next_index < len(sorted_lists[list_index]):
            next_value = sorted_lists[list_index][next_index]
            heapq.heappush(heap, (next_value, list_index, next_index))
    return result

lists = [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
print(merge_k_sorted_lists(lists))

### Watch different list lengths and empty lists

In [ ]:
lists = [[1, 10], [], [2, 3, 11, 15], [4, 5, 6]]
print(merge_k_sorted_lists(lists))

### Why is the merge O(N log k)?

`N` is the total number of values; `k` is the number of lists. The heap contains at most one candidate per list, so its size is at most `k`. Each value is pushed and popped at most once, with O(log k) heap operations: total time **O(N log k)**. The heap uses **O(k)** extra space, not counting the O(N) output.

Combining and sorting costs O(N log N). The heap approach can save work when `k` is much smaller than `N`.

Python also provides `heapq.merge()`, a convenient lazy merge for already-sorted inputs.

In [ ]:
lists = [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
print(list(heapq.merge(*lists)))

## 7. Tests: verify the functions

Assertions help us catch mistakes. If this cell runs without an error, these expected results passed.

In [ ]:
assert top_k_frequent([4, 4, 4, 2, 2, 7, 7, 7, 7, 9, 9, 1], 2) == [(4, 3), (7, 4)] or set(top_k_frequent([4, 4, 4, 2, 2, 7, 7, 7, 7, 9, 9, 1], 2)) == {(4, 4), (3, 4)}
assert merge_k_sorted_lists([[1, 4, 7], [2, 5, 8], [3, 6, 9]]) == [1, 2, 3, 4, 5, 6, 7, 8, 9]
assert merge_k_sorted_lists([[], [1], [], [0, 2]]) == [0, 1, 2]
assert merge_k_sorted_lists([]) == []
print("All tests passed.")

## 8. Common mistakes

1. **Thinking a heap is sorted:** it is not; only the parent-child rule is guaranteed.
2. **Claiming a heap is always faster:** for one minimum, `min()` is already O(n).
3. **Confusing peek and pop:** `heap[0]` reads; `heappop()` removes.
4. **Ignoring empty heaps:** `heap[0]` fails if the heap is empty.
5. **Mixing up largest and most frequent:** they answer different questions.
6. **Ignoring ties:** define a tie rule if the task needs one.
7. **Merging unsorted lists:** heap merge assumes each input list is sorted.

## 9. Class challenges

Try each one before checking the code. Explain not just what the output is, but why.

### Challenge A: find the minimum two ways

In [ ]:
values = [8, 4, 6, 2, 9]
print("min():", min(values))
heap = values.copy()
heapq.heapify(heap)
print("heap root:", heap[0])

### Challenge B: find the two smallest using a heap

In [ ]:
values = [10, 3, 8, 1, 5, 2]
heap = values.copy()
heapq.heapify(heap)
print([heapq.heappop(heap) for _ in range(2)])

### Challenge C: merge these lists

In [ ]:
print(merge_k_sorted_lists([[1, 7], [2, 4, 8], [3, 5, 6]]))

## 10. A simple 60-second teaching script

1. Show the unsorted list and find its minimum with `min()`.
2. Build a heap with `heapq.heapify()` and show that index `0` is the minimum.
3. Explain that a heap is not sorted; only its root is guaranteed to be smallest.
4. Be honest about Big-O: `min()` is O(n), and building a heap is O(n), so a heap is not automatically faster for one minimum.
5. Demonstrate repeated `heappop()` calls or merging sorted lists to show where heaps help.
6. Summarize: `heapify` O(n), root lookup O(1), push/pop O(log n), merge k sorted lists O(N log k).

**Final idea:** choose the simplest tool that fits the task. Heaps shine when you need repeated minimums, a limited set of best candidates, or an efficient merge of sorted lists.